# Matching GPX ↔ PCS — Toutes les années

Ce notebook apparie les fichiers GPX (parcours des courses, issus de la-flamme-rouge.eu)
avec les fichiers CSV de résultats (issus de procyclingstats.com).

Le matching repose sur trois informations clés : **la date** (jour/mois/année),
**le numéro d'étape** (ou 'result' pour une course d'un jour), et **le nom normalisé** de la course.

Le processus se déroule en deux passes successives avec des seuils de similarité décroissants
pour maximiser le taux de couverture tout en limitant les faux positifs.

In [1]:
import os
import re
import glob
import pandas as pd
import subprocess

# Installation des dépendances de fuzzy matching si absentes
subprocess.run(['pip', 'install', 'thefuzz', 'python-Levenshtein'], capture_output=True)
from thefuzz import fuzz

# ── Chemins ──────────────────────────────────────────────────────────────────
# GPX_DIR      : dossier contenant tous les fichiers .gpx téléchargés depuis la-flamme-rouge.eu
# PCS_BASE_DIR : dossier racine des CSV ProcyclingStats, organisés par sous-dossiers d'année
# OUTPUT_BASE  : dossier de sortie pour les CSV de matching
GPX_DIR      = '/Users/arthurdeletang/Desktop/Stage M1/Code/data/gpx_files_2'
PCS_BASE_DIR = '/Users/arthurdeletang/Desktop/Stage M1/Code/data/pcs'
OUTPUT_BASE  = '/Users/arthurdeletang/Desktop/Stage M1/Code/data/matching'

# Création des sous-dossiers de sortie (all = toutes catégories, wt = World Tour,
# 1uwt = 1.UWT courses d'un jour, 2uwt = 2.UWT courses par étapes)
for d in ['all', 'wt', '1uwt', '2uwt']:
    os.makedirs(os.path.join(OUTPUT_BASE, d), exist_ok=True)

# Années à traiter
ANNEES = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# Chargement du calendrier de courses GPX (généré par le notebook La_Flamme_Rouge)
# Colonnes : name, id, day, month, year
races_df = pd.read_csv('/Users/arthurdeletang/Desktop/Stage M1/Code/data/races.csv')
all_races = races_df.values.tolist()

# Correspondance mois anglais -> numéro, utilisé pour parser les noms de fichiers PCS
mois_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8,
    'September': 9, 'October': 10, 'November': 11, 'December': 12
}

print(f'Races totales chargées : {len(all_races)}')
print(f'Années à traiter : {ANNEES}')

/Users/arthurdeletang/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/arthurdeletang/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Races totales chargées : 15475
Années à traiter : [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [2]:
# ============================================================
# FONCTIONS UTILITAIRES ET MAPPINGS DE NOMS
# ============================================================

def normaliser(nom):
    """
    Normalise un nom de course pour le rendre comparable entre les deux sources.
    Convertit en minuscules et remplace tout caractère non alphanumérique par un tiret,
    puis supprime les tirets multiples ou en bordure.
    Exemple : 'Tour de France 2024' -> 'tour-de-france-2024'
    """
    nom = nom.lower()
    nom = re.sub(r'[^a-z0-9]', '-', nom)
    nom = re.sub(r'-+', '-', nom)
    return nom.strip('-')

def parser_pcs(nom_fichier):
    """
    Extrait les métadonnées d'un fichier CSV ProcyclingStats à partir de son nom.
    Format attendu : 'jour_Mois_annee__stage__nom-course_annee__categorie.csv'
    Exemple : '15_March_2023__stage_3__paris-nice_2023__2.UWT.csv'
    Retourne un dict avec les clés : jour, mois, annee, stage, nom, categorie.
    """
    parts = nom_fichier.replace('.csv', '').split('__')
    date_parts = parts[0].split('_')
    jour = int(date_parts[0])
    mois = mois_map[date_parts[1]]
    annee = int(date_parts[2])
    stage_info = parts[1]
    nom_course = parts[2].rsplit('_', 1)[0]
    categorie = parts[3]
    return {
        'jour': jour, 'mois': mois, 'annee': annee,
        'stage': stage_info, 'nom': nom_course, 'categorie': categorie
    }

# ── Mapping global PCS -> GPX ─────────────────────────────────────────────────
# Les deux sources (PCS et la-flamme-rouge) n'utilisent pas toujours le même nom
# pour une même course. Ce dictionnaire contient les correspondances stables
# (valables sur toutes les années).
# Clé :  PCS  |  Valeur :  GPX
mapping_noms_pcs = {
    'tour-auvergne-rhone-alpes':      'criterium-du-dauphine',
    'e3-harelbeke':                   'e3-saxo-classic',
    'great-ocean-road-race':          'cadel-evans-great-ocean-road-race',
    'vuelta-a-espana':                'la-vuelta-ciclista-a-espana',
    'volta-a-catalunya':              'volta-ciclista-a-catalunya',
    'tour-down-under':                'santos-tour-down-under',
    'gp-quebec':                      'grand-prix-cycliste-de-quebec',
    'gp-montreal':                    'grand-prix-cycliste-de-montreal',
    'tour-of-guangxi':                'gree-tour-of-guangxi',
    'san-sebastian':                  'donostia-san-sebastian-klasikoa',
    'cyclassics-hamburg':             'bemer-cyclassics',
    'race-torquay':                   'surf-coast-classic',
    'deia-trophy':                    'trofeo-serra-tramuntana',
    'gp-d-ouverture':                 'grand-prix-cycliste-de-marseille-la-marseillaise',
    'istrian-spring-tour':            'istarsko-proljece-istrian-spring-trophy',
    'gp-izola-butan-plin':            'gp-slovenian-istria',
    'tour-of-turkey':                 'presidential-cycling-tour-of-turkiye',
    'gp-de-plumelec':                 'grand-prix-du-morbihan',
    'tour-of-konya':                  'tour-of-sakarya',
    'albani-classic-fyen-rundt':      'fyn-rundt-tour-of-funen',
    'heist-op-den-berg':              'heylen-vastgoed-heistse-pijl',
    'course-cycliste-de-solidarnosc': 'course-cycliste-de-solidarnosc-et-des-champions-olympiques',
    'volta-a-portugal':               'volta-a-portugal-em-bicicleta',
    'gp-impanis-van-petegem':         'super-8-classic',
    'tour-of-croatia':                'cro-race',
    'vuelta-internacional-a-costa-rica': 'vuelta-ciclistica-internacional-a-costa-rica',
    'baloise-belgium-tour':           'baloise-belgium-tour',
    'memorial-frank-vandenbroucke':   'binche-chimay-binche',
    'nc-united-states':               'usa-road-national-championship',
    'japan-cup':                      'utsunomiya-japan-cup-road-race',
    'panamerican-championships':      'copaci-campeonato-panamericano-de-ruta',
    'panamerican-champ-itt':          'copaci-campeonato-panamericano-de-ruta-itt',
    'tour-of-belgium':                'baloise-belgium-tour',
    'tour-of-magnificent-qinghai':    'tour-of-qinghai-lake',
    'tour-of-samsun':                 'tour-of-routhe-salvation',
    'gp-du-canton-d-argovie':         'grosser-preis-des-kantons-aargau',
    'banyuwangi-tour-de-ijen':        'tour-de-banyuwangi-ijen',
    'giro-di-toscana':                'giro-della-toscana-memorial-alfredo-martini',
    'tour-du-poitou-charentes-et-de-la-vienne': 'tour-poitou-charentes-en-nouvelle-aquitaine',
    'tour-bitwa-warszawska-1920':     'tour-battle-of-warsaw',
    'hadeland-gp':                    'sundvolden-gp',
    'cholet-pays-de-loire':           'cholet-agglo-tour',
    'trofeu-da-arrabida':             'classica-da-arrabida-cyclin-portugal',
    'banja-luka-belgrade-i':          'belgrade-banjaluka',
    'giro-di-romagna':                'giro-della-romagna',
    'alanya-cup':                     'tour-of-alanya',
    'south-aegean-tour':              'visit-south-aegean-islands',
    'tour-cycliste-international-la-provence': 'tour-de-la-provence',
    'gp-de-valence':                  'classica-comunitat-valenciana-1969-gran-premi-valencia',
    'circuit-des-xi-villes':          'elfstedenronde-brugge',
    'grand-prix-erciyes-mimar-sinan-me': 'grand-prix-yildizdagi',
    'visegrad-4-bicycle-race-gp-hungary': 'visegrad-4-kerekparverseny',
    'gp-de-la-ville-de-nogent-sur-oise': 'grand-prix-international-d-isbergues',
    'trofeo-torino-biella-giro-della-provincia-di-biell': 'giro-della-provincia-di-biella-torino-biella',
    'trofeo-pollenca-port-d-andratx': 'trofeo-andratx-mirador-des-colomer-pollenca',
    'alula-tour':                     'saudi-tour',
    'volta-nxt-classic':              'volta-limburg-classic',
    'vuelta-a-ecuador':               'vuelta-ciclistica-al-ecuador',
    'gp-int-torres-vedras':           'gp-internacional-torres-vedras-trofeu-joaquim-agostinho',
    'oberosterreichrundfahrt':        'osterreich-rundfahrt',
    'tour-du-pays-de-montbeliard':    'tour-du-pays-de-montbeliard',
    'gp-stad-zottegem':               'gp-stad-zottegem',
    'grand-prix-altinkale':           'grand-prix-altinkale',
    'pan-american-games':             'juegos-panamericanos-santiago',
    'pan-american-games---tt':        'juegos-panamericanos-santiago-itt',
    'tour-of-hellas':                 'tour-of-hellas',
    'sibiu-cycling-tour':             'sibiu-cycling-tour',
    'tour-de-hokkaido':               'tour-de-hokkaido',
    'grote-prijs-marcel-kint':        'grote-prijs-marcel-kint',
    'east-midlands-international-cicle-classic': 'rutland-melton-cicle-classic',
    'gp-internacional-beiras-e-serra-da-estrela': 'grande-premio-beiras-e-serra-da-estrela',
    'gran-camino':                    'o-gran-camino-the-historical-route',
    'rhodes-gp':                      'tour-of-rhodes-powered-by-rodos-palace',
    'la-drome-classic':               'faun-drome-classic',
    'tour-of-norway':                 'tour-of-norway',
    'rhone-alpes-isere-tour':         'alpes-isere-tour',
    'boucles-de-la-mayenne':          'boucles-de-la-mayenne-credit-mutuel',
    'tour-de-langkawi':               'petronas-le-tour-de-langkawi',
    'munsterland-giro':               'sparkassen-muensterland-giro',
    'giro-dell-emilia':               'giro-dell-emilia',
    'coppa-agostoni':                 'coppa-agostoni-giro-delle-brianze',
    'oita-urban-classic':             'oita-urban-classic-road-race',
    'circuito-de-getxo':              'circuito-de-getxo-memorial-hermanos-otxoa',
    'syedra-ancient-city':            'grand-prix-syedra-ancient-city',
    'paris-camembert':                'paris-camembert',
    'route-adelie-de-vitre':          'la-route-adelie-de-vitre',
    'famenne-ardenne-classic':        'lotto-famenne-ardenne-classic',
    'gp-antalya':                     'grand-prix-antalya',
    'grand-prix-aspendos':            'grand-prix-aspendos',
    'grand-prix-kahramanmaras':       'grand-prix-kahramanmaras',
    'grand-prix-edebiyat-yolu':       'grand-prix-edebiyat-yolu',
    'trofeo-citta-di-brescia':        'trofeo-citta-di-brescia-mem-rino-fiori',
    'giro-del-medio-brenta':          'giro-del-medio-brenta',
    'midden-brabant-poort-omloop':    'midden-brabant-poort-omloop',
    'kreiz-breizh-elites':            'kreiz-breizh-elites',
    'vuelta-pilsen-a-colombia':       'vuelta-a-colombia',
    'vuelta-a-guatemala':             'vuelta-a-guatemala',
    'boucle-de-l-artois':             'boucle-de-l-artois',
    'kuurne-brussel-kuurne':          'kuurne-brussel-kuurne',
    'omloop-van-het-waasland':        'omloop-van-het-waasland',
    'fleche-ardennaise':              'fleche-ardennaise',
    'gp-herning':                     'grand-prix-herning',
    'tour-of-holland':                'nibc-tour-of-holland',
    'okolo-slovenska':                'tour-of-slovaquie',
    'tour-of-austria':                'osterreich-rundfahrt',
    'renewi-tour':                    'binckbank-tour',
    'gp-citta-di-peccioli':           'coppa-sabatini',
    'vuelta-ciclista-a-la-provincia-de-san-juan': 'vuelta-a-san-juan-internacional',
    'eschborn-frankfurt':             'eschborn-frankfurt-rund-um-den-finanzplatz',
    'five-rings-of-moscow':           'five-rings-of-moscow',
    'hammer-hong-kong':               'hammer-series-hammer-hong-kong',
    'omloop-het-nieuwsblad-beloften': 'omloop-het-nieuwsblad-beloften',
    'paris-roubaix-u23':              'paris-roubaix-espoirs',
    'szlakiem-walk-majora-hubala':    'szlakiem-walk-majora-hubala',
    'tour-de-hongrie':                'tour-de-hongrie',
    'tour-of-rhodes':                 'international-tour-of-rhodes',
    'tour-of-szeklerland':            'cycling-tour-of-szeklerland',
    'volta-ao-alentejo':              'volta-ao-alentejo',
    'region-pays-de-la-loire':        'circuit-cycliste-sarthe-pays-de-la-loire',
    'grand-prix-international-de-la-pharmacie-centrale-': 'tour-cycliste-international-de-la-pharmacie-centrale-de-tunisie',
    'tour-du-cameroun':               'tour-du-cameroun',
    'tour-of-rwanda':                 'tour-of-rwanda',
    'chrono-des-nations':             'chrono-des-nations',
}

# ── Mapping spécifique à une année ───────────────────────────────────────────
# Certaines courses changent de nom d'une année à l'autre (ex. rechampionnats,
# rebranding de sponsor). Ce dictionnaire prend le pas sur mapping_noms_pcs
# pour les couples (nom_pcs, annee) concernés.
mapping_par_annee = {
    # San Juan : nom différent de la version multi-années dans mapping_noms_pcs
    ('vuelta-ciclista-a-la-provincia-de-san-juan', 2017): 'vuelta-ciclista-a-la-provincia-de-san-juan',
    ('vuelta-ciclista-a-la-provincia-de-san-juan', 2018): 'vuelta-ciclista-a-la-provincia-de-san-juan',
    ('vuelta-ciclista-a-la-provincia-de-san-juan', 2020): 'vuelta-ciclista-a-la-provincia-de-san-juan',
    # E3 Harelbeke : sponsor différent en 2017-2018
    ('e3-harelbeke', 2017): 'record-bank-e3-harelbeke',
    ('e3-harelbeke', 2018): 'record-bank-e3-harelbeke',
    # Eschborn-Frankfurt : même nom sur la-flamme-rouge pour 2018-2025
    ('eschborn-frankfurt', 2018): 'eschborn-frankfurt',
    ('eschborn-frankfurt', 2019): 'eschborn-frankfurt',
    ('eschborn-frankfurt', 2020): 'eschborn-frankfurt',
    ('eschborn-frankfurt', 2021): 'eschborn-frankfurt',
    ('eschborn-frankfurt', 2022): 'eschborn-frankfurt',
    ('eschborn-frankfurt', 2023): 'eschborn-frankfurt',
    ('eschborn-frankfurt', 2024): 'eschborn-frankfurt',
    ('eschborn-frankfurt', 2025): 'eschborn-frankfurt',
    # Circuit des Ardennes
    ('circuit-des-ardennes', 2018): 'circuit-des-ardennes-international',
    # Rhodes GP : nom change chaque année
    ('rhodes-gp', 2018): 'international-rhodes-grand-prix',
    ('rhodes-gp', 2019): 'rhodes-grand-prix',
    ('rhodes-gp', 2020): 'rhodes-grand-prix',
    ('rhodes-gp', 2021): 'rhodes-grand-prix',
    ('rhodes-gp', 2022): 'rhodes-grand-prix',
    ('rhodes-gp', 2023): 'rhodes-gp-by-culture-sports-organization',
    ('rhodes-gp', 2024): 'tour-of-rhodes-powered-by-rodos-palace',
    ('rhodes-gp', 2025): 'tour-of-rhodes-powered-by-rodos-palace',
    # Okolo Slovenska
    ('okolo-slovenska', 2019): 'okolo-slovenska-tour-de-slovaquie',
    ('okolo-slovenska', 2020): 'tour-de-slovaquie',
    ('okolo-slovenska', 2021): 'okolo-slovenska',
    ('okolo-slovenska', 2022): 'okolo-slovenska',
    ('okolo-slovenska', 2023): 'okolo-slovenska',
    ('okolo-slovenska', 2024): 'okolo-slovenska',
    ('okolo-slovenska', 2025): 'okolo-slovenska',
    # Tour of Austria
    ('tour-of-austria', 2017): 'osterreich-rundfahrt',
    ('tour-of-austria', 2018): 'osterreich-rundfahrt',
    ('tour-of-austria', 2019): 'osterreich-rundfahrt',
    ('tour-of-austria', 2020): 'osterreich-rundfahrt',
    ('tour-of-austria', 2021): 'osterreich-rundfahrt',
    ('tour-of-austria', 2022): 'osterreich-rundfahrt',
    ('tour-of-austria', 2023): 'tour-of-austria',
    ('tour-of-austria', 2024): 'tour-of-austria',
    ('tour-of-austria', 2025): 'tour-of-austria',
    # Renewi Tour
    ('renewi-tour', 2023): 'renewi-tour',
    ('renewi-tour', 2024): 'renewi-tour',
    ('renewi-tour', 2025): 'renewi-tour',
    # Tour du Cameroun
    ('tour-du-cameroun', 2021): 'tour-de-cameroon',
    # Tour du Poitou-Charentes
    ('tour-du-poitou-charentes-et-de-la-vienne', 2017): 'tour-poitou-charentes',
    # Istrian Spring Tour
    ('istrian-spring-tour', 2019): 'istarsko-proljece-istrian-sprint-trophy',
    # Tour of Rwanda
    ('tour-of-rwanda', 2025): 'tour-du-rwanda',
    # Trofeu da Arrabida
    ('trofeu-da-arrabida', 2025): 'trofeu-internacional-da-arrabida',
    # Tour of Szeklerland
    ('tour-of-szeklerland', 2019): 'cycling-tour-of-szeklerland',
    ('tour-of-szeklerland', 2020): 'cycling-tour-of-szeklerland',
    ('tour-of-szeklerland', 2021): 'cycling-tour-of-szeklerland',
    ('tour-of-szeklerland', 2022): 'cycling-tour-of-szeklerland',
    ('tour-of-szeklerland', 2023): 'cycling-tour-of-szeklerland',
    ('tour-of-szeklerland', 2024): 'tour-of-szeklerland',
    # Tour of Holland
    ('tour-of-holland', 2024): 'tour-of-holland',
    ('tour-of-holland', 2025): 'nibc-tour-of-holland',
    ('itzulia-basque-country', 2017): 'vuelta-al-pais-vasco',
    ('eschborn-frankfurt', 2017): 'Eschborn-Frankfurt Rund um den Finanzplatz',
    ('tour-of-croatia', 2017): 'tour-of-croatia',
    ('volta-a-portugal', 2018): 'volta-a-portugal-em-bicicleta-santander-totta',

}

# ── Décalages de numéro d'étape ───────────────────────────────────────────────
# PCS et la-flamme-rouge ne numérotent pas toujours les étapes de la même façon.
# Un offset de -1 signifie que le stage_N de PCS correspond au stage_(N-1) de LFR
# (typiquement quand LFR inclut un prologue ou un jour de repos que PCS n'indexe pas).
stage_offset = {
    ('tour-de-suisse', 2017):                    -1,
    ('tour-de-suisse', 2024):                    -1,
    ('tour-of-japan', 2024):                     -1,
    ('tour-of-japan', 2023):                     -1,
    ('tour-of-austria', 2024):                   -1,
    ('tour-alsace', 2021):                       -1,
    ('tour-alsace', 2022):                       -1,
    ('tour-alsace', 2023):                       -1,
    ('tour-alsace', 2024):                       -1,
    ('tour-alsace', 2025):                       -1,
    ('tour-of-szeklerland', 2024):               -1,
    ('istrian-spring-tour', 2024):                1,
    ('volta-a-portugal', 2024):                  -1,
    ('fleche-du-sud', 2025):                     -1,
    ('tour-cycliste-international-la-provence', 2019): -1,
    ('itzulia-basque-country', 2022):            -1,
    ('tour-de-la-mirabelle', 2022):              -1,
    ('tour-du-pays-de-montbeliard', 2022):       -1,
    ('gp-int-torres-vedras', 2022):              -1,
    ('gp-int-torres-vedras', 2023):              -1,
    ('okolo-jiznich-cech', 2022):                -1,
    ('oberosterreichrundfahrt', 2023):           -1,
    ('tour-of-rhodes', 2023):                    -1,

}

# ── Corrections manuelles ciblées ─────────────────────────────────────────────
# Pour certains cas très spécifiques (ex. jour de repos non indexé dans LFR
# qui décale les étapes à partir d'un certain point), on force manuellement
# la correspondance GPX pour une clé (jour, mois, annee, stage) donnée.
corrections_manuelles = {
    # BinckBank 2020 : LFR considère le jour de repos comme Stage 2,
    # donc Stage 3 PCS correspond à Stage 2 LFR, etc.
    (1, 10, 2020, 'stage_3'): [('2020 BinckBank Tour Stage 2.gpx', 'binckbank-tour')],
    (2, 10, 2020, 'stage_4'): [('2020 BinckBank Tour Stage 3.gpx', 'binckbank-tour')],
    (3, 10, 2020, 'stage_5'): [('2020 BinckBank Tour Stage 4.gpx', 'binckbank-tour')],
    # Tour Poitou-Charentes : stage_4 et stage_5 PCS = stage_3 et stage_4 LFR
    # (car LFR split l'étape 3 en 3-1 et 3-2 que PCS compte comme deux étapes séparées)
    (24, 8, 2017, 'stage_4'): [('2017 Tour Poitou-Charentes Stage 3-1.gpx', 'tour-poitou-charentes')],
    (25, 8, 2017, 'stage_5'): [('2017 Tour Poitou-Charentes Stage 4.gpx',   'tour-poitou-charentes')],
    (23, 8, 2018, 'stage_4'): [('2018 Tour Poitou-Charentes en Nouvelle Aquitaine Stage 3-1.gpx', 'tour-poitou-charentes-en-nouvelle-aquitaine')],
    (24, 8, 2018, 'stage_5'): [('2018 Tour Poitou-Charentes en Nouvelle Aquitaine Stage 4.gpx',   'tour-poitou-charentes-en-nouvelle-aquitaine')],
    (29, 8, 2019, 'stage_4'): [('2019 Tour Poitou-Charentes en Nouvelle Aquitaine Stage 3-1.gpx', 'tour-poitou-charentes-en-nouvelle-aquitaine')],
    (30, 8, 2019, 'stage_5'): [('2019 Tour Poitou-Charentes en Nouvelle Aquitaine Stage 4.gpx',   'tour-poitou-charentes-en-nouvelle-aquitaine')],
    (26, 8, 2021, 'stage_4'): [('2021 Tour Poitou - Charentes en Nouvelle Aquitaine Stage 3-1.gpx', 'tour-poitou---charentes-en-nouvelle-aquitaine')],
    (27, 8, 2021, 'stage_5'): [('2021 Tour Poitou - Charentes en Nouvelle Aquitaine Stage 4.gpx',   'tour-poitou---charentes-en-nouvelle-aquitaine')],
    (25, 8, 2022, 'stage_4'): [('2022 Tour Poitou - Charentes en Nouvelle Aquitaine Stage 3-1.gpx', 'tour-poitou---charentes-en-nouvelle-aquitaine')],
    (26, 8, 2022, 'stage_5'): [('2022 Tour Poitou - Charentes en Nouvelle Aquitaine Stage 4.gpx',   'tour-poitou---charentes-en-nouvelle-aquitaine')],
    # course-cycliste-de-solidarnosc : Stage 1-1 et 1-2 le même jour → décalage à partir de stage_2
    # 2018
    ( 4, 7, 2018, 'stage_2'): [('2018 Course Cycliste de Solidarnosc et des Champions Olympiques Stage 1-2.gpx', 'course-cycliste-de-solidarnosc-et-des-champions-olympiques')],
    ( 5, 7, 2018, 'stage_3'): [('2018 Course Cycliste de Solidarnosc et des Champions Olympiques Stage 2.gpx',   'course-cycliste-de-solidarnosc-et-des-champions-olympiques')],
    ( 6, 7, 2018, 'stage_4'): [('2018 Course Cycliste de Solidarnosc et des Champions Olympiques Stage 3.gpx',   'course-cycliste-de-solidarnosc-et-des-champions-olympiques')],
    ( 7, 7, 2018, 'stage_5'): [('2018 Course Cycliste de Solidarnosc et des Champions Olympiques Stage 4.gpx',   'course-cycliste-de-solidarnosc-et-des-champions-olympiques')],
    # 2021
    (23, 6, 2021, 'stage_2'): [('2021 Course Cycliste de Solidarnosc et des Champions Olympiques Stage 1-2.gpx', 'course-cycliste-de-solidarnosc-et-des-champions-olympiques')],
    (24, 6, 2021, 'stage_3'): [('2021 Course Cycliste de Solidarnosc et des Champions Olympiques Stage 2.gpx',   'course-cycliste-de-solidarnosc-et-des-champions-olympiques')],
    (25, 6, 2021, 'stage_4'): [('2021 Course Cycliste de Solidarnosc et des Champions Olympiques Stage 3.gpx',   'course-cycliste-de-solidarnosc-et-des-champions-olympiques')],
    (26, 6, 2021, 'stage_5'): [('2021 Course Cycliste de Solidarnosc et des Champions Olympiques Stage 4.gpx',   'course-cycliste-de-solidarnosc-et-des-champions-olympiques')],
    # 2023
    (28, 6, 2023, 'stage_2'): [('2023 Course Cycliste de Solidarnosc et des Champions Olympiques Stage 1-2.gpx', 'course-cycliste-de-solidarnosc-et-des-champions-olympiques')],
    (29, 6, 2023, 'stage_3'): [('2023 Course Cycliste de Solidarnosc et des Champions Olympiques Stage 2.gpx',   'course-cycliste-de-solidarnosc-et-des-champions-olympiques')],
    (30, 6, 2023, 'stage_4'): [('2023 Course Cycliste de Solidarnosc et des Champions Olympiques Stage 3.gpx',   'course-cycliste-de-solidarnosc-et-des-champions-olympiques')],
    ( 1, 7, 2023, 'stage_5'): [('2023 Course Cycliste de Solidarnosc et des Champions Olympiques Stage 4.gpx',   'course-cycliste-de-solidarnosc-et-des-champions-olympiques')],
    # 2024 : Stage 1-1 et 1-2 le 26/6 + Rest day le 27/6 → stage_2 PCS = 27/6 rest day skippé
    (27, 6, 2024, 'stage_2'): [('2024 Course Cycliste de Solidarnosc et des Champions Olympiques Stage 1-2.gpx', 'course-cycliste-de-solidarnosc-et-des-champions-olympiques')],
    # istrian-spring-tour 2019 : pas de prologue LFR, tout est décalé
    (14, 3, 2019, 'prologue'): [('2019 Istarsko Proljece - Istrian Sprint Trophy Stage 1.gpx', 'istarsko-proljece-istrian-sprint-trophy')],
    (15, 3, 2019, 'stage_1'): [('2019 Istarsko Proljece - Istrian Sprint Trophy Stage 2.gpx', 'istarsko-proljece-istrian-sprint-trophy')],
    (16, 3, 2019, 'stage_2'): [('2019 Istarsko Proljece - Istrian Sprint Trophy Stage 3.gpx', 'istarsko-proljece-istrian-sprint-trophy')],
    (17, 3, 2019, 'stage_3'): [('2019 Istarsko Proljece - Istrian Sprint Trophy Stage 4.gpx', 'istarsko-proljece-istrian-sprint-trophy')],

    # istrian-spring-tour 2024 : pas de prologue LFR
    ( 7, 3, 2024, 'prologue'): [('2024 Istarsko Proljece - Istrian Spring Trophy Stage 1.gpx', 'istarsko-proljece-istrian-spring-trophy')],
    # istrian-spring-tour 2021 : nom GPX différent + prologue LFR = stage_1 PCS
    (11, 3, 2021, 'stage_1'): [('2021 Istrian Spring Trophy Prologue.gpx', 'istrian-spring-trophy')],
    (12, 3, 2021, 'stage_2'): [('2021 Istrian Spring Trophy Stage 1.gpx', 'istrian-spring-trophy')],
    (13, 3, 2021, 'stage_3'): [('2021 Istrian Spring Trophy Stage 2.gpx', 'istrian-spring-trophy')],
    (14, 3, 2021, 'stage_4'): [('2021 Istrian Spring Trophy Stage 3.gpx', 'istrian-spring-trophy')],

    # tour-alsace 2018
    (1, 8, 2018, 'prologue'): [('2018 Tour Alsace Stage 1.gpx', 'tour-alsace')],
    (2, 8, 2018, 'stage_1'):  [('2018 Tour Alsace Stage 2.gpx', 'tour-alsace')],
    (3, 8, 2018, 'stage_2'):  [('2018 Tour Alsace Stage 3.gpx', 'tour-alsace')],
    (4, 8, 2018, 'stage_3'):  [('2018 Tour Alsace Stage 4.gpx', 'tour-alsace')],
    (5, 8, 2018, 'stage_4'):  [('2018 Tour Alsace Stage 5.gpx', 'tour-alsace')],

    # tour-alsace 2019
    (31, 7, 2019, 'prologue'): [('2019 Tour Alsace Stage 1.gpx', 'tour-alsace')],
    (2,  8, 2019, 'stage_2'):  [('2019 Tour Alsace Stage 3.gpx', 'tour-alsace')],
    (3,  8, 2019, 'stage_3'):  [('2019 Tour Alsace Stage 4.gpx', 'tour-alsace')],
    (4,  8, 2019, 'stage_4'):  [('2019 Tour Alsace Stage 5.gpx', 'tour-alsace')],

    # tour-of-austria 2017 : pas de prologue PCS, Stage 1 PCS = Stage 1 LFR
    (3, 7, 2017, 'stage_1'): [('2017 Osterreich Rundfahrt Stage 1.gpx', 'osterreich-rundfahrt')],
    (4, 7, 2017, 'stage_2'): [('2017 Osterreich Rundfahrt Stage 2.gpx', 'osterreich-rundfahrt')],
    (5, 7, 2017, 'stage_3'): [('2017 Osterreich Rundfahrt Stage 3.gpx', 'osterreich-rundfahrt')],
    (6, 7, 2017, 'stage_4'): [('2017 Osterreich Rundfahrt Stage 4.gpx', 'osterreich-rundfahrt')],
    (7, 7, 2017, 'stage_5'): [('2017 Osterreich Rundfahrt Stage 5.gpx', 'osterreich-rundfahrt')],
    (8, 7, 2017, 'stage_6'): [('2017 Osterreich Rundfahrt Stage 6.gpx', 'osterreich-rundfahrt')],

    # tour-des-alpes-maritimes = Tour du Haut Var sur LFR
    (18, 2, 2017, 'stage_1'): [('2017 Tour Cycliste International du Haut Var-matin Stage 1.gpx', 'tour-cycliste-international-du-haut-var-matin')],
    (19, 2, 2017, 'stage_2'): [('2017 Tour Cycliste International du Haut Var-matin Stage 2.gpx', 'tour-cycliste-international-du-haut-var-matin')],
    (17, 2, 2018, 'stage_1'): [('2018 50eme Tour Cycliste International du Haut Var Matin Stage 1.gpx', '50eme-tour-cycliste-international-du-haut-var-matin')],
    (18, 2, 2018, 'stage_2'): [('2018 50eme Tour Cycliste International du Haut Var Matin Stage 2.gpx', '50eme-tour-cycliste-international-du-haut-var-matin')],
    (22, 2, 2019, 'stage_1'): [('2019 51eme tour cycliste international du Haut Var Stage 1.gpx', '51eme-tour-cycliste-international-du-haut-var')],
    (23, 2, 2019, 'stage_2'): [('2019 51eme tour cycliste international du Haut Var Stage 2.gpx', '51eme-tour-cycliste-international-du-haut-var')],
    (24, 2, 2019, 'stage_3'): [('2019 51eme tour cycliste international du Haut Var Stage 3.gpx', '51eme-tour-cycliste-international-du-haut-var')],

    # volta-a-portugal 2018 : GPX indexés en mois 7, PCS en mois 8
    ( 1, 8, 2018, 'prologue'): [('2018 Volta a Portugal em Bicicleta Santander Totta Prologue.gpx', 'volta-a-portugal-em-bicicleta-santander-totta')],
    ( 2, 8, 2018, 'stage_1'):  [('2018 Volta a Portugal em Bicicleta Santander Totta Stage 1.gpx',   'volta-a-portugal-em-bicicleta-santander-totta')],
    ( 3, 8, 2018, 'stage_2'):  [('2018 Volta a Portugal em Bicicleta Santander Totta Stage 2.gpx',   'volta-a-portugal-em-bicicleta-santander-totta')],
    ( 4, 8, 2018, 'stage_3'):  [('2018 Volta a Portugal em Bicicleta Santander Totta Stage 3.gpx',   'volta-a-portugal-em-bicicleta-santander-totta')],
    ( 5, 8, 2018, 'stage_4'):  [('2018 Volta a Portugal em Bicicleta Santander Totta Stage 4.gpx',   'volta-a-portugal-em-bicicleta-santander-totta')],

    # volta-a-portugal 2019
    (31, 7, 2019, 'prologue'): [('2019 Volta a Portugal Santander Prologue.gpx', 'volta-a-portugal-santander')],
    ( 2, 8, 2019, 'stage_2'):  [('2019 Volta a Portugal Santander Stage 2.gpx',  'volta-a-portugal-santander')],
    ( 3, 8, 2019, 'stage_3'):  [('2019 Volta a Portugal Santander Stage 3.gpx',  'volta-a-portugal-santander')],
    ( 4, 8, 2019, 'stage_4'):  [('2019 Volta a Portugal Santander Stage 4.gpx',  'volta-a-portugal-santander')],
}

# ── Mapping pays pour championnats nationaux (NC) ─────────────────────────────
# Pour les courses de catégorie 'NC' (National Championship), on filtre les GPX
# candidats par pays. Ce dict fait le lien entre le code pays dans le slug PCS
# et le mot-clé attendu dans le nom du fichier GPX.
pays_mapping = {
    'australia': 'australian', 'austria': 'austrian', 'austria2': 'austrian',
    'belgium': 'belgian', 'canada': 'canadian', 'canada2': 'canadian',
    'china': 'chinese', 'croatia': 'croatian', 'czech': 'czech',
    'denmark': 'danish', 'dominican-republic': 'dominican', 'france': 'french',
    'georgia': 'georgian', 'germany': 'german', 'ireland': 'irish',
    'israel': 'israel', 'italy': 'italian', 'japan': 'japanese',
    'luxembourg': 'luxembourg', 'norway': 'norwegian', 'poland': 'polish',
    'portugal': 'portuguese', 'romania': 'romanian', 'serbia': 'serbian',
    'slovakia': 'slovakian', 'slovenia': 'slovenian', 'spain': 'spanish',
    'ukraine': 'ukrainian', 'usa': 'usa', 'united-states': 'usa',
    'new-zealand': 'new-zealand', 'colombia': 'colombian', 'ecuador': 'ecuador',
    'south-africa': 'south-africa', 'uruguay': 'uruguay', 'malaysia': 'malaysia',
    'philippines': 'philippines', 'thailand': 'thailand', 'india': 'indian',
}

# ── Mapping championnats continentaux ─────────────────────────────────────────
# Pour les championnats continentaux, on filtre les GPX candidats en vérifiant
# que le nom du fichier contient l'un des mots-clés du continent.
continents = {
    'central-american': ['central-american', 'caribbean', 'america'],
    'asian': ['asian', 'asia'],
    'european': ['european', 'europe', 'uec'],
    'african': ['african', 'africa', 'cac'],
    'oceania': ['oceania', 'occ'],
    'panamerican': ['panamerican', 'copaci', 'pan-american'],
}

# Mots-clés à exclure des candidats GPX pour éviter les faux positifs
# (courses féminines, U23, espoirs, etc. qui partagent la même date que la course élite)
exclure = ['femmes', 'ladies', 'femenina', 'next-gen', 'u23', 'under-23',
           'women', 'dames', 'femminile', 'espoirs', 'juniors', 'rest-day',
           'feminin', 'bloom']

# Mots-clés identifiant les courses World Tour dans les noms GPX
# (utilisé pour des filtrages optionnels, non actif dans le pipeline principal)
wt_keywords = [
    'tour-down-under', 'cadel-evans', 'uae-tour', 'nieuwsblad', 'strade-bianche',
    'paris-nice', 'tirreno-adriatico', 'milano-sanremo', 'catalunya', 'e3-saxo',
    'gent-wevelgem', 'dwars-door-vlaanderen', 'ronde-van-vlaanderen', 'itzulia-basque',
    'paris-roubaix', 'amstel-gold', 'fleche-wallonne', 'liege-bastogne', 'tour-de-romandie',
    'eschborn-frankfurt', 'giro-d-italia', 'dauphine', 'tour-de-suisse', 'tour-de-france',
    'san-sebastian', 'donostia', 'tour-de-pologne', 'cyclassics', 'renewi-tour',
    'vuelta-ciclista-a-espana', 'bretagne-classic', 'quebec', 'montreal', 'lombardia', 'guangxi'
]

def extraire_continent_pcs(nom_pcs):
    """
    Détermine si une course PCS est un championnat continental,
    et si oui retourne le continent correspondant.
    Retourne None si la course n'est pas un championnat continental.
    """
    for continent, keywords in continents.items():
        if any(kw in nom_pcs for kw in keywords):
            return continent
    return None

def extraire_pays_pcs(nom_pcs):
    """
    Extrait le code pays depuis un slug de championnat national PCS.
    Gère deux préfixes possibles : 'nc-' et 'national-championships-'.
    Supprime les suffixes de format (-itt, -rr, -road-race, etc.).
    Retourne None si la course n'est pas un championnat national.
    """
    if nom_pcs.startswith('nc-'):
        pays = nom_pcs[3:]
        pays = re.sub(r'-itt$|-rr$|-road-race$|-ttt$|-me-road-race$|-me-itt$|-me-ttt$', '', pays)
        return pays
    if nom_pcs.startswith('national-championships-'):
        pays = nom_pcs[len('national-championships-'):]
        pays = re.sub(r'-itt$|-rr$|-road-race$|-me-ttt$|-me-itt$|-me-road-race$', '', pays)
        return pays
    return None

def nom_pcs_vers_gpx(nom_pcs, annee=None):
    """
    Traduit un nom de course PCS en son équivalent GPX.
    Priorité au mapping par année (plus spécifique),
    puis au mapping global. Retourne le nom PCS inchangé si aucun mapping trouvé.
    """
    if annee and (nom_pcs, annee) in mapping_par_annee:
        return mapping_par_annee[(nom_pcs, annee)]
    return mapping_noms_pcs.get(nom_pcs, nom_pcs)

def appliquer_offset(stage, nom_pcs, annee):
    """
    Corrige le numéro d'étape PCS pour l'aligner avec la numérotation LFR.
    Si un offset est défini pour (nom_pcs, annee), ajoute cet offset au numéro
    d'étape. Un résultat <= 0 est converti en 'prologue'.
    Ne modifie pas les stages non numériques ('result', 'prologue').
    """
    offset_key = (nom_pcs, annee)
    if offset_key not in stage_offset:
        return stage
    if not stage.startswith('stage_'):
        return stage
    offset = stage_offset[offset_key]
    stage_num = int(re.search(r'\d+', stage).group(0))
    new_stage_num = stage_num + offset
    if new_stage_num <= 0:
        return 'prologue'
    return f'stage_{new_stage_num}'

print('✅ Fonctions utilitaires et mappings définis')

✅ Fonctions utilitaires et mappings définis


In [3]:
# ============================================================
# FONCTIONS DE MATCHING
# ============================================================

def construire_dict_races(races, annee):
    """
    Construit un dictionnaire indexant les fichiers GPX disponibles
    par clé (jour, mois, annee, stage).

    Pour chaque fichier GPX, extrait la date et le numéro d'étape depuis
    le nom (ex. '2023 Tour de France Stage 5.gpx' -> stage='stage_5').
    Les courses d'un seul jour reçoivent stage='result'.

    Applique ensuite les corrections manuelles pour l'année en cours.

    Retourne : dict[(jour, mois, annee, stage)] -> [(nom_fichier, nom_norme), ...]
    """
    dict_races = {}
    for race in races:
        nom_fichier = str(race[0]).replace('"', '') + '.gpx'
        stage_match = re.search(r'Stage (\d+)', str(race[0]))
        prologue = 'Prologue' in str(race[0])
        if stage_match:
            stage = f'stage_{stage_match.group(1)}'
        elif prologue:
            stage = 'prologue'
        else:
            stage = 'result'
        # Retire l'année (4 premiers caractères) et les mentions 'Stage N' ou 'Prologue'
        nom_base = str(race[0])[5:]
        nom_base = re.sub(r' Stage \d+', '', nom_base)
        nom_base = re.sub(r' Prologue', '', nom_base)
        nom_norm = normaliser(nom_base)
        cle = (int(race[2]), int(race[3]), int(race[4]), stage)
        if cle not in dict_races:
            dict_races[cle] = []
        if (nom_fichier, nom_norm) not in dict_races[cle]:
            dict_races[cle].append((nom_fichier, nom_norm))
    # Ecrase les entrées concernées par des corrections manuelles
    for cle, val in corrections_manuelles.items():
        if cle[2] == annee:
            dict_races[cle] = val
    return dict_races


def construire_dict_pcs(pcs_dir):
    """
    Construit un dictionnaire indexant les fichiers CSV PCS
    par clé (jour, mois, annee, stage).

    Lit tous les fichiers .csv du dossier et parse leur nom via parser_pcs.
    Une même clé peut contenir plusieurs courses (ex. deux championnats
    nationaux différents le même jour).

    Retourne : dict[(jour, mois, annee, stage)] -> [{'fichier', 'nom', 'categorie'}, ...]
    """
    dict_pcs = {}
    if not os.path.exists(pcs_dir):
        return dict_pcs
    for f in os.listdir(pcs_dir):
        if f.endswith('.csv'):
            try:
                info = parser_pcs(f)
                stage = info['stage'].replace(' ', '_').lower()
                cle = (info['jour'], info['mois'], info['annee'], stage)
                if cle not in dict_pcs:
                    dict_pcs[cle] = []
                dict_pcs[cle].append({'fichier': f, 'nom': info['nom'], 'categorie': info['categorie']})
            except Exception:
                pass
    return dict_pcs


def calculer_candidats(pcs_info, cle_gpx, gpx_utilises, dict_races):
    """
    Détermine la liste des fichiers GPX candidats pour une course PCS donnée.

    Filtre successivement :
    1. Retire les GPX déjà attribués à une autre course (gpx_utilises).
    2. Retire les GPX contenant des mots-clés exclus (femmes, U23, etc.).
    3. Pour les championnats nationaux (NC) : ne garde que les GPX contenant
       le mot-clé 'national', puis filtre par pays si possible.
    4. Pour les championnats continentaux : filtre par mots-clés du continent.
    5. Si aucun candidat ne reste après filtrage (et que la course n'est pas NC
       ou continentale, et n'a pas de mapping explicite), élargit sans filtre
       de nom (hors GPX déjà utilisés).

    Retourne : liste de tuples (nom_fichier, nom_norme) des GPX candidats.
    """
    if cle_gpx not in dict_races:
        return []
    nom_pcs_original = pcs_info['nom']
    annee = pcs_info.get('annee')
    candidats_gpx = dict_races[cle_gpx]
    # Filtre de base : GPX déjà utilisés et mots-clés exclus
    candidats_gpx_filtres = []
    for f, n in candidats_gpx:
        if f in gpx_utilises:
            continue
        if any(ex in n for ex in exclure):
            continue
        if pcs_info['categorie'] == 'NC' and 'national' not in n:
            continue
        candidats_gpx_filtres.append((f, n))
    # Filtre pays pour les championnats nationaux
    if pcs_info['categorie'] == 'NC' and candidats_gpx_filtres:
        pays = extraire_pays_pcs(nom_pcs_original)
        if pays:
            mot_cle_pays = pays_mapping.get(pays)
            if mot_cle_pays:
                filtres_pays = [(f, n) for f, n in candidats_gpx_filtres if mot_cle_pays in n]
                candidats_gpx_filtres = filtres_pays
            else:
                candidats_gpx_filtres = []
    # Filtre continent pour les championnats continentaux
    est_continental = extraire_continent_pcs(nom_pcs_original) is not None
    continent = extraire_continent_pcs(nom_pcs_original)
    if continent and candidats_gpx_filtres:
        keywords_continent = continents[continent]
        filtres_continent = [(f, n) for f, n in candidats_gpx_filtres
                             if any(kw in n for kw in keywords_continent)]
        candidats_gpx_filtres = filtres_continent if filtres_continent else []
    # Fallback sans filtre de nom si aucun candidat et pas de contrainte forte
    a_un_mapping = (nom_pcs_original in mapping_noms_pcs or
                    (annee and (nom_pcs_original, annee) in mapping_par_annee))
    if not candidats_gpx_filtres and pcs_info['categorie'] != 'NC' and not est_continental and not a_un_mapping:
        candidats_gpx_filtres = [(f, n) for f, n in candidats_gpx if f not in gpx_utilises]
    return candidats_gpx_filtres


def faire_passe(dict_pcs, dict_races, gpx_utilises, matches_existants, seuil):
    """
    Effectue une passe de matching entre les fichiers PCS et GPX.

    Pour chaque course PCS non encore matchée :
    1. Traduit son nom via nom_pcs_vers_gpx.
    2. Corrige le numéro d'étape via appliquer_offset.
    3. Récupère les candidats GPX via calculer_candidats.
    4. Calcule le score de similarité (fuzz.partial_ratio) entre le nom PCS
       traduit et chaque candidat GPX.
    5. Retient le meilleur score si celui-ci dépasse le seuil.
    6. Marque le GPX retenu comme utilisé pour éviter les doublons.

    Les courses sont traitées dans l'ordre chronologique pour stabiliser
    l'attribution en cas de conflit (un même GPX ne peut être attribué
    qu'à une seule course PCS).

    Retourne : dict des nouveaux matches {cle_unique -> {'gpx', 'pcs', 'nom_pcs', 'categorie', 'score'}}
    """
    nouveaux_matches = {}
    # Tri chronologique pour un traitement déterministe
    pcs_tries = sorted(dict_pcs.items(), key=lambda x: (x[0][2], x[0][1], x[0][0]))
    for cle, pcs_list in pcs_tries:
        jour, mois, annee, stage = cle
        for pcs_info in pcs_list:
            nom_pcs_original = pcs_info['nom']
            cle_unique = (jour, mois, annee, stage, nom_pcs_original)
            if cle_unique in matches_existants:
                continue
            # Traduction du nom et correction du numéro d'étape
            nom_pcs_mapped = nom_pcs_vers_gpx(nom_pcs_original, annee)
            stage_gpx = appliquer_offset(stage, nom_pcs_original, annee)
            cle_gpx = (jour, mois, annee, stage_gpx)
            pcs_info_annee = {**pcs_info, 'annee': annee}
            candidats = calculer_candidats(pcs_info_annee, cle_gpx, gpx_utilises, dict_races)
            if not candidats:
                continue
            # Sélection du meilleur candidat par score de similarité
            best_score = 0
            best_gpx = None
            for nom_fichier, nom_gpx in candidats:
                score = fuzz.partial_ratio(nom_pcs_mapped, nom_gpx)
                if score > best_score:
                    best_score = score
                    best_gpx = nom_fichier
            if best_score >= seuil:
                nouveaux_matches[cle_unique] = {
                    'gpx': best_gpx, 'pcs': pcs_info['fichier'],
                    'nom_pcs': nom_pcs_original, 'categorie': pcs_info['categorie'],
                    'score': best_score
                }
                gpx_utilises.add(best_gpx)
    return nouveaux_matches

print('✅ Fonctions de matching définies')

✅ Fonctions de matching définies


In [4]:
# ============================================================
# MATCHING POUR TOUTES LES ANNÉES
# ============================================================
# Pour chaque année :
#   - Passe 1 (seuil 90) : matches de haute confiance
#   - Passe 2 (seuil 60) : matches plus incertains sur les cas restants
# Les GPX attribués en passe 1 ne sont plus disponibles en passe 2.
# Sortie : 4 CSV par année (all, wt, 1uwt, 2uwt)
# ============================================================

for ANNEE in ANNEES:
    separateur = '='*50
    print(f'\n{separateur}')
    print(f'Traitement {ANNEE}...')

    pcs_dir = os.path.join(PCS_BASE_DIR, str(ANNEE))
    if not os.path.exists(pcs_dir):
        print(f'  Dossier PCS {ANNEE} introuvable, on passe.')
        continue

    races = [r for r in all_races if int(r[4]) == ANNEE]
    dict_races = construire_dict_races(races, ANNEE)
    dict_pcs = construire_dict_pcs(pcs_dir)
    print(f'  Races GPX : {len(dict_races)} | PCS : {len(dict_pcs)}')

    gpx_utilises = set()  # Ensemble des GPX déjà attribués (partagé entre les deux passes)
    matches = {}

    # Passe 1 : seuil élevé (90) -> matches très fiables
    m1 = faire_passe(dict_pcs, dict_races, gpx_utilises, matches, seuil=90)
    matches.update(m1)

    # Passe 2 : seuil bas (60) -> récupère les cas difficiles non traités en passe 1
    m2 = faire_passe(dict_pcs, dict_races, gpx_utilises, matches, seuil=60)
    matches.update(m2)

    print(f'  Matches : {len(matches)} (passe1={len(m1)}, passe2={len(m2)})')

    # ── CSV toutes courses (matchées + non matchées) ──────────────────────────
    rows_all = []
    cles_matchees = set(matches.keys())
    for cle, val in matches.items():
        rows_all.append({
            'jour': cle[0], 'mois': cle[1], 'annee': cle[2], 'stage': cle[3],
            'nom_pcs': val['nom_pcs'], 'categorie': val['categorie'],
            'fichier_gpx': val['gpx'], 'fichier_pcs': val['pcs'],
            'score': val['score'], 'statut': 'match'
        })
    # Ajoute les courses PCS sans GPX trouvé (statut 'pcs_only')
    for cle, pcs_list in dict_pcs.items():
        jour, mois, annee, stage = cle
        for pcs_info in pcs_list:
            cle_unique = (jour, mois, annee, stage, pcs_info['nom'])
            if cle_unique in cles_matchees:
                continue
            rows_all.append({
                'jour': jour, 'mois': mois, 'annee': annee, 'stage': stage,
                'nom_pcs': pcs_info['nom'], 'categorie': pcs_info['categorie'],
                'fichier_gpx': None, 'fichier_pcs': pcs_info['fichier'],
                'score': None, 'statut': 'pcs_only'
            })
    df_all = pd.DataFrame(rows_all).sort_values(['annee', 'mois', 'jour']).reset_index(drop=True)
    df_all.to_csv(os.path.join(OUTPUT_BASE, 'all', f'matching_{ANNEE}.csv'), index=False)

    # ── CSV World Tour (toutes UWT, puis filtres 1.UWT et 2.UWT) ─────────────
    rows_wt = []
    cles_wt = set()
    for cle, val in matches.items():
        if 'UWT' not in val['categorie']:
            continue
        rows_wt.append({
            'jour': cle[0], 'mois': cle[1], 'annee': cle[2], 'stage': cle[3],
            'nom_pcs': val['nom_pcs'], 'categorie': val['categorie'],
            'fichier_gpx': val['gpx'], 'fichier_pcs': val['pcs'],
            'score': val['score'], 'statut': 'match'
        })
        cles_wt.add((cle[0], cle[1], cle[2], cle[3], cle[4]))
    for cle, pcs_list in dict_pcs.items():
        jour, mois, annee, stage = cle
        for pcs_info in pcs_list:
            if 'UWT' not in pcs_info['categorie']:
                continue
            cle_unique = (jour, mois, annee, stage, pcs_info['nom'])
            if cle_unique in cles_wt:
                continue
            rows_wt.append({
                'jour': jour, 'mois': mois, 'annee': annee, 'stage': stage,
                'nom_pcs': pcs_info['nom'], 'categorie': pcs_info['categorie'],
                'fichier_gpx': None, 'fichier_pcs': pcs_info['fichier'],
                'score': None, 'statut': 'pcs_only'
            })
            cles_wt.add(cle_unique)

    df_wt = pd.DataFrame(rows_wt).sort_values(['annee', 'mois', 'jour']).reset_index(drop=True)
    df_wt.to_csv(os.path.join(OUTPUT_BASE, 'wt', f'matching_wt_{ANNEE}.csv'), index=False)

    df_1uwt = df_wt[df_wt['categorie'] == '1.UWT'].reset_index(drop=True)
    df_2uwt = df_wt[df_wt['categorie'] == '2.UWT'].reset_index(drop=True)
    df_1uwt.to_csv(os.path.join(OUTPUT_BASE, '1uwt', f'matching_1uwt_{ANNEE}.csv'), index=False)
    df_2uwt.to_csv(os.path.join(OUTPUT_BASE, '2uwt', f'matching_2uwt_{ANNEE}.csv'), index=False)

    print(f'  ✅ {ANNEE} — all:{len(df_all)} | wt:{len(df_wt)} | 1uwt:{len(df_1uwt)} | 2uwt:{len(df_2uwt)}')

print('\n✅ Matching termine pour toutes les annees !')


Traitement 2017...
  Races GPX : 706 | PCS : 849
  Matches : 595 (passe1=497, passe2=98)
  ✅ 2017 — all:1266 | wt:177 | 1uwt:20 | 2uwt:157

Traitement 2018...
  Races GPX : 1137 | PCS : 854
  Matches : 1040 (passe1=884, passe2=156)
  ✅ 2018 — all:1250 | wt:178 | 1uwt:20 | 2uwt:158

Traitement 2019...
  Races GPX : 1145 | PCS : 837
  Matches : 1062 (passe1=893, passe2=169)
  ✅ 2019 — all:1223 | wt:180 | 1uwt:21 | 2uwt:159

Traitement 2020...
  Races GPX : 531 | PCS : 388
  Matches : 448 (passe1=366, passe2=82)
  ✅ 2020 — all:535 | wt:111 | 1uwt:11 | 2uwt:100

Traitement 2021...
  Races GPX : 845 | PCS : 604
  Matches : 700 (passe1=596, passe2=104)
  ✅ 2021 — all:868 | wt:145 | 1uwt:16 | 2uwt:129

Traitement 2022...
  Races GPX : 952 | PCS : 673
  Matches : 813 (passe1=746, passe2=67)
  ✅ 2022 — all:1024 | wt:146 | 1uwt:19 | 2uwt:127

Traitement 2023...
  Races GPX : 1045 | PCS : 786
  Matches : 937 (passe1=868, passe2=69)
  ✅ 2023 — all:1165 | wt:162 | 1uwt:20 | 2uwt:142

Traitement 20

In [5]:
# ============================================================
# DIAGNOSTIC : COURSES PCS SANS MATCH GPX
# ============================================================
# Agrège les CSV de toutes les années et affiche les courses PCS
# pour lesquelles aucun GPX n'a été trouvé (statut 'pcs_only'),
# en excluant les championnats nationaux (NC) et continentaux (CC)
# dont l'absence de GPX est souvent normale.
# Le CSV de sortie permet d'identifier les courses à ajouter
# manuellement dans les mappings si nécessaire.
# ============================================================

all_dfs = []
for f in sorted(glob.glob(os.path.join(OUTPUT_BASE, 'all', 'matching_*.csv'))):
    all_dfs.append(pd.read_csv(f))

df_toutes_annees = pd.concat(all_dfs).reset_index(drop=True)

# Filtre : statut 'pcs_only' et hors catégories contenant 'C' (NC, CC, 1.Ncup...)
pcs_only_toutes = df_toutes_annees[
    (df_toutes_annees['statut'] == 'pcs_only') &
    (~df_toutes_annees['categorie'].str.contains('C', na=False))
][['jour', 'mois', 'annee', 'stage', 'nom_pcs', 'categorie', 'fichier_pcs']].sort_values(['annee', 'mois', 'jour']).reset_index(drop=True)

print(f'Total PCS sans GPX (hors CC/NC) : {len(pcs_only_toutes)}')
pcs_only_toutes.to_csv(os.path.join(OUTPUT_BASE, 'pcs_only_all_years.csv'), index=False)
print(pcs_only_toutes.to_string())

Total PCS sans GPX (hors CC/NC) : 755
     jour  mois  annee     stage                                             nom_pcs categorie                                                                              fichier_pcs
0      13     1   2017   stage_1                                   vuelta-al-tachira       2.2                                13_January_2017__stage_1__vuelta-al-tachira_2017__2.2.csv
1      14     1   2017   stage_2                                   vuelta-al-tachira       2.2                                14_January_2017__stage_2__vuelta-al-tachira_2017__2.2.csv
2      15     1   2017   stage_3                                   vuelta-al-tachira       2.2                                15_January_2017__stage_3__vuelta-al-tachira_2017__2.2.csv
3      16     1   2017   stage_4                                   vuelta-al-tachira       2.2                                16_January_2017__stage_4__vuelta-al-tachira_2017__2.2.csv
4      17     1   2017   stage_5          

In [6]:
df_all = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(os.path.join(OUTPUT_BASE, 'all', 'matching_*.csv')))]).reset_index(drop=True)
df_nm  = df_all[df_all['statut'] == 'pcs_only'].copy()

# ── Non-matchés par année ───────────────────────────────────
print("=== Non-matchés par année ===")
print(df_nm.groupby('annee').size().to_string())

# ── Non-matchés par catégorie ───────────────────────────────
print("\n=== Non-matchés par catégorie ===")
print(df_nm.groupby('categorie').size().sort_values(ascending=False).to_string())

# ── Noms les plus fréquents (hors NC/CC) ────────────────────
print("\n=== Noms les plus fréquents (hors NC/CC) — top 30 ===")
df_nm_pro = df_nm[~df_nm['categorie'].str.contains('C', na=False)]
print(df_nm_pro.groupby('nom_pcs').size().sort_values(ascending=False).head(30).to_string())

=== Non-matchés par année ===
annee
2017    671
2018    210
2019    161
2020     87
2021    168
2022    211
2023    228
2024    239
2025    205

=== Non-matchés par catégorie ===
categorie
NC          1397
2.2          469
1.2          134
2.1           72
JR            27
1.2U          15
2.2U          13
1.1           10
2.HC           8
JC             7
1.HC           7
CC             6
2.Ncup         4
2.Pro          4
2.UWT          2
1.Ncup         2
1.UWT          1
1.Pro          1
Olympics       1

=== Noms les plus fréquents (hors NC/CC) — top 30 ===
nom_pcs
tour-du-cameroun                    37
tour-du-maroc                       16
tour-alsace                         14
tour-de-cote-d-ivoire               13
tour-des-pays-de-savoie             12
vuelta-pilsen-a-colombia            12
vuelta-ciclista-a-venezuela         10
vuelta-al-tachira                   10
tour-du-faso                        10
five-rings-of-moscow                10
tour-de-korea                      

In [7]:
import os
import xml.etree.ElementTree as ET
from tqdm import tqdm

gpx_dir = GPX_DIR
manquent_elevation = []
fichiers = [f for f in os.listdir(gpx_dir) if f.endswith('.gpx')]

for fname in tqdm(fichiers):
    try:
        tree = ET.parse(os.path.join(gpx_dir, fname))
        root = tree.getroot()
        ns = {'gpx': 'http://www.topografix.com/GPX/1/1'}
        ele = root.find('.//gpx:ele', ns)
        if ele is None:
            manquent_elevation.append(fname)
    except Exception as e:
        manquent_elevation.append(f'{fname} (erreur: {e})')

print(f'Total GPX : {len(fichiers)}')
print(f'Sans élévation : {len(manquent_elevation)}')
for f in manquent_elevation:
    print(f'  {f}')

100%|██████████| 11306/11306 [08:40<00:00, 21.73it/s]

Total GPX : 11306
Sans élévation : 142
  2022 National Road Championships - United Arab Emirates - ITT.gpx (erreur: not well-formed (invalid token): line 9, column 121)
  2020 Cyprus Road National Championships - ITT.gpx (erreur: not well-formed (invalid token): line 9, column 121)
  2017 Tour de l'Avenir Rest day.gpx (erreur: not well-formed (invalid token): line 9, column 121)
  2023 La Vuelta Ciclista a Espana Rest day.gpx (erreur: not well-formed (invalid token): line 9, column 121)
  2019 Hongkonger Road National Championships - ITT.gpx (erreur: not well-formed (invalid token): line 9, column 121)
  2018 Giro d'Italia Rest day.gpx (erreur: not well-formed (invalid token): line 9, column 121)
  2019 Georgian Road National Championships.gpx (erreur: not well-formed (invalid token): line 9, column 121)
  2021 Giro d'Italia Rest day.gpx (erreur: not well-formed (invalid token): line 9, column 121)
  2019 Marocaine Road National Championships - ITT.gpx (erreur: not well-formed (invalid

In [8]:
import pandas as pd
import os

# Prends un fichier au hasard
pcs_dir = os.path.join(PCS_BASE_DIR, '2023')
f = os.listdir(pcs_dir)[0]
print(f)
df = pd.read_csv(os.path.join(pcs_dir, f))
print(df.head())
print(df.columns.tolist())

10_August_2023__stage_1__tour-of-szeklerland_2023__2.2.csv
  Rnk    GC Timelag  BIB  H2H Specialty  Age  \
0   1  16.0   +0:25  163  NaN    Sprint   20   
1   2  10.0   +0:18  162  NaN    Sprint   19   
2   3  37.0   +0:38  112  NaN    Sprint   23   
3   4  20.0   +0:30   61  NaN    Sprint   23   
4   5  27.0   +0:34  116  NaN    Sprint   23   

                                        Rider                     Team  UCI  \
0         Skerl DanielCycling Team Friuli ASD  Cycling Team Friuli ASD  7.0   
1  Bruttomesso AlbertoCycling Team Friuli ASD  Cycling Team Friuli ASD  3.0   
2                     Buda SimoneSolme - Olmo             Solme - Olmo  1.0   
3            Proć BartłomiejSantic - Wibatech        Santic - Wibatech  NaN   
4                    Berzi AndreaSolme - Olmo             Solme - Olmo  NaN   

   Pnt Unnamed: 11            Time            date  classification  \
0  5.0         10″  2:34:062:34:06  10 August 2023             2.2   
1  2.0          6″          ,,0:00  1